# param-grad-access — faded example 3: put the NaN check first in grad health

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `param-grad-access`. Running the beacon reports progress on the `PyTorch: param.grad access` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: param.grad access` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`param-grad-access`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "param-grad-access"
DD_SUBTOPIC = "PyTorch: param.grad access"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

Classifying grad health must test NaN/Inf BEFORE the zero check, because an all-NaN tensor would otherwise slip past the zero test ambiguously. The first branch flags `nan` on any NaN or Inf element.

## Faded exercise 3

Complete `classify_grad_health` by adding the NaN/Inf detection branch that must run first. Fill in the nan/inf condition.

**Fill in:** the NaN-or-Inf detection condition that must be checked first

In [ ]:
import torch as t
import torch.nn as nn

t.manual_seed(5)

def classify_grad_health(model):
    out = {}
    for name, p in model.named_parameters():
        if p.grad is None:
            continue
        is_bad = t.isnan(p.grad).any().item() or t.isinf(p.grad).any().item()
        if is_bad:
            out[name] = 'nan'
        elif p.grad.abs().max().item() == 0.0:
            out[name] = 'zero'
        else:
            out[name] = 'ok'
    return out

m = nn.Linear(2, 1)
m(t.randn(3, 2)).sum().backward()
print(classify_grad_health(m))


def _test():
    m = nn.Sequential(nn.Linear(2, 2), nn.Linear(2, 2))
    ps = list(m.parameters())
    ps[0].grad = t.tensor([[float('nan'), 1.0], [2.0, 3.0]])
    ps[1].grad = t.zeros(2)
    ps[2].grad = t.ones(2, 2)
    ps[3].grad = t.full((2,), float('inf'))
    labels = list(classify_grad_health(m).values())
    # order: weight0=nan, bias0=zero, weight1=ok, bias1=nan(inf)
    assert labels == ['nan', 'zero', 'ok', 'nan']


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t
import torch.nn as nn

t.manual_seed(5)

def classify_grad_health(model):
    out = {}
    for name, p in model.named_parameters():
        if p.grad is None:
            continue
        is_bad = t.isnan(p.grad).any().item() or t.isinf(p.grad).any().item()
        if is_bad:
            out[name] = 'nan'
        elif p.grad.abs().max().item() == 0.0:
            out[name] = 'zero'
        else:
            out[name] = 'ok'
    return out

m = nn.Linear(2, 1)
m(t.randn(3, 2)).sum().backward()
print(classify_grad_health(m))
```
</details>